## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations. See Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> for how to set up a separate project for your work.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...<br/>And if you post about it on LinkedIn and tag me, then I'll weigh in to amplify your achievement. If you see other students posting, please give them your encouragement too.
            </span>
        </td>
    </tr>
</table>

In [2]:
# Start with imports - ask the Cursor Agent to explain any package that you don't know
# Claude uses the anthropic library. Gemini's endpoint is OpenAI compatible, so it uses the openai library.

import os
import json
import anthropic
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [3]:
# Always remember to do this!
load_dotenv(override=True)

True

In [4]:
# Print the key prefixes to help with any debugging

anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:2]}")
else:
    print("Gemini API Key not set")


Anthropic API Key exists and begins sk-ant-
Gemini API Key exists and begins AQ


In [5]:
request = """
Please come up with a challenging, nuanced question with a succinct answer,
that I can ask a number of LLMs to evaluate their intelligence.
Not a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.
Include in your question that the answer must be short.
"""
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [6]:
messages

[{'role': 'user',
  'content': '\nPlease come up with a challenging, nuanced question with a succinct answer,\nthat I can ask a number of LLMs to evaluate their intelligence.\nNot a mathematical puzzle, but more of a thought-provoking question that requires intelligent insight.\nInclude in your question that the answer must be short.\nAnswer only with the question, no explanation.'}]

In [7]:
# Create the Claude client and a small helper. Claude returns a list of content blocks; we join the text ones.

claude = anthropic.Anthropic()

def ask_claude(messages, model="claude-opus-5-5", max_tokens=4096):
    response = claude.messages.create(model=model, max_tokens=max_tokens, messages=messages)
    return "".join(block.text for block in response.content if block.type == "text")

question = ask_claude(messages)
display(Markdown(question))

A foundation decides to fund only research projects that expert reviewers judge to have at least a 90% chance of succeeding, believing this will maximize scientific progress. Why will this policy likely produce fewer transformative breakthroughs than funding projects judged to have only a 10% chance of success? Answer in one sentence.

## Calling LLMs from two providers

We will compare Claude and Gemini.
Gemini provides an API endpoint that is compatible with OpenAI, as explained in Guide 9 in the guides folder.
So we can simply use that endpoint as if we are using OpenAI, while Claude uses Anthropic's own library.

In [8]:
# Gemini's OpenAI compatible URL

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [9]:
# OpenAI client library with Gemini's base_url and key
# If this surprises you, please see Guide 9 in the Guides folder!

gemini = OpenAI(api_key=gemini_api_key, base_url=GEMINI_BASE_URL)

In [10]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

In [11]:
def record(model_name, answer):
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [12]:
model_name = "claude-opus-5-5"

answer = ask_claude(messages, model=model_name)

record(model_name, answer)

Projects that experts can confidently rate as 90% likely to succeed are, almost by definition, ones whose outcomes current knowledge already predicts, so they yield incremental confirmations. Transformative breakthroughs are precisely the results that defy expert expectations and so get rated as long shots, and because scientific payoffs are heavy-tailed, a few rare successes among high-risk bets can outweigh many safe, modest wins.

In [13]:
model_name = "gemini-3.1-flash-lite"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

By prioritizing projects with near-certain outcomes, the foundation incentivizes incremental work and minimizes the risk-taking essential for the radical discoveries that drive major scientific paradigm shifts.

In [15]:
# So where are we?

print(len(competitors))
print(competitors)
print(answers)


2
['claude-opus-5-5', 'gemini-3.1-flash-lite']
['Projects that experts can confidently rate as 90% likely to succeed are, almost by definition, ones whose outcomes current knowledge already predicts, so they yield incremental confirmations. Transformative breakthroughs are precisely the results that defy expert expectations and so get rated as long shots, and because scientific payoffs are heavy-tailed, a few rare successes among high-risk bets can outweigh many safe, modest wins.', 'By prioritizing projects with near-certain outcomes, the foundation incentivizes incremental work and minimizes the risk-taking essential for the radical discoveries that drive major scientific paradigm shifts.']


In [16]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: claude-opus-5-5

Projects that experts can confidently rate as 90% likely to succeed are, almost by definition, ones whose outcomes current knowledge already predicts, so they yield incremental confirmations. Transformative breakthroughs are precisely the results that defy expert expectations and so get rated as long shots, and because scientific payoffs are heavy-tailed, a few rare successes among high-risk bets can outweigh many safe, modest wins.
Competitor: gemini-3.1-flash-lite

By prioritizing projects with near-certain outcomes, the foundation incentivizes incremental work and minimizes the risk-taking essential for the radical discoveries that drive major scientific paradigm shifts.


In [17]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [18]:
print(together)

# Response from competitor 1

Projects that experts can confidently rate as 90% likely to succeed are, almost by definition, ones whose outcomes current knowledge already predicts, so they yield incremental confirmations. Transformative breakthroughs are precisely the results that defy expert expectations and so get rated as long shots, and because scientific payoffs are heavy-tailed, a few rare successes among high-risk bets can outweigh many safe, modest wins.

# Response from competitor 2

By prioritizing projects with near-certain outcomes, the foundation incentivizes incremental work and minimizes the risk-taking essential for the radical discoveries that drive major scientific paradigm shifts.




In [19]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [20]:
print(judge)

You are judging a competition between 2 competitors.
Each model has been given this question:

A foundation decides to fund only research projects that expert reviewers judge to have at least a 90% chance of succeeding, believing this will maximize scientific progress. Why will this policy likely produce fewer transformative breakthroughs than funding projects judged to have only a 10% chance of success? Answer in one sentence.

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}

Here are the responses from each competitor:

# Response from competitor 1

Projects that experts can confidently rate as 90% likely to succeed are, almost by definition, ones whose outcomes current knowledge already predicts, so they yield incremental confirmations. Transforma

In [21]:
judge_messages = [{"role": "user", "content": judge}]

## And now for the judge!

Claude will judge the responses. The competitors are shown only by number, but Claude is also one of them, so treat the ranking as a learning exercise.

In [22]:
# Judgement time!

results = ask_claude(judge_messages)
print(results)


{"results": ["1", "2"]}


In [23]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: claude-opus-5-5
Rank 2: gemini-3.1-flash-lite


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>